In [1]:
import os
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm  # Progress bar
import warnings

# Suppress warnings (librosa can be chatty)
warnings.filterwarnings('ignore')

# CONFIGURATION
PROCESSED_DIR = "../data/processed"
METADATA_PATH = os.path.join(PROCESSED_DIR, "metadata.csv")

# Audio Config
SAMPLE_RATE = 22050
DURATION = 4  # We fix audio at 4 seconds. 
              # (Most RAVDESS/TESS clips are 2-4s. 5s adds too much silence).
SAMPLES_PER_TRACK = SAMPLE_RATE * DURATION

print("Ready to extract features.")

Ready to extract features.


In [2]:
def extract_features(file_path):
    try:
        # 1. Load Audio
        y, sr = librosa.load(file_path, sr=SAMPLE_RATE)

        # 2. Pad or Truncate to Fixed Length (4 seconds)
        if len(y) > SAMPLES_PER_TRACK:
            y = y[:SAMPLES_PER_TRACK]
        else:
            padding = SAMPLES_PER_TRACK - len(y)
            y = np.pad(y, (0, padding), mode='constant')

        # 3. Extract Features (Mean of the frames to get 1D vector)
        
        # MFCC (Texture)
        mfcc = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40).T, axis=0)
        
        # Chroma (Pitch)
        stft = np.abs(librosa.stft(y))
        chroma = np.mean(librosa.feature.chroma_stft(S=stft, sr=sr).T, axis=0)
        
        # Mel Spectrogram (Frequency)
        mel = np.mean(librosa.feature.melspectrogram(y=y, sr=sr).T, axis=0)
        
        # Contrast (Spectral peaks vs valleys)
        contrast = np.mean(librosa.feature.spectral_contrast(S=stft, sr=sr).T, axis=0)
        
        # Tonnetz (Tonal centroids)
        tonnetz = np.mean(librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr).T, axis=0)
        
        # Combine all features into one 1D array
        return np.hstack([mfcc, chroma, mel, contrast, tonnetz])

    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Test on one random file to check shape
test_df = pd.read_csv(METADATA_PATH)
random_file = test_df.iloc[0]['path']
feat = extract_features(random_file)
print(f"Feature Vector Shape: {feat.shape}")
# Expected: (40 + 12 + 128 + 7 + 6) = ~193 features

Feature Vector Shape: (193,)


In [3]:
# Load Metadata
df = pd.read_csv(METADATA_PATH)

X = []
y = []

print(f"Starting feature extraction for {len(df)} files...")

# Iterate with progress bar
for index, row in tqdm(df.iterrows(), total=len(df)):
    features = extract_features(row['path'])
    
    if features is not None:
        X.append(features)
        y.append(row['emotion'])

print("\nExtraction Complete.")

Starting feature extraction for 12162 files...


100%|██████████| 12162/12162 [45:03<00:00,  4.50it/s] 


Extraction Complete.
